# Topics Covered

# Cache

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_8")\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 12:41:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [7]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_1gb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [4]:
#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('product_name',StringType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products.csv',schema=product_schema_struct,header=True)

df_product.show(5)

[Stage 1:>                                                          (0 + 1) / 1]

+----------+--------------------+----------+
|product_id|        product_name|cost_price|
+----------+--------------------+----------+
|         1|      Wireless Mouse|     120.0|
|         2| Mechanical Keyboard|     135.0|
|         3|       USB C Charger|     150.0|
|         4|        Laptop Stand|     165.0|
|         5|Noise Cancelling ...|     180.0|
+----------+--------------------+----------+
only showing top 5 rows



In [ ]:
df_orders.filter(col('state')=='Odisha').write.mode('overwrite').format('noop').save()

In [9]:
df_orders.cache() # after cache all the process will be faster,cache will be in memory now

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [10]:
#without any action ,cache wont be stored
df_orders.count()

20000000

# mode of cache

In [12]:
from pyspark import StorageLevel


In [13]:
df_orders.persist(StorageLevel.MEMORY_ONLY)

26/03/06 13:06:20 WARN CacheManager: Asked to cache already cached data.


DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

# first clear the cache



In [14]:
df_orders.unpersist()

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [16]:
# now go for persist

df_orders.persist(StorageLevel.MEMORY_ONLY) # memory serialised

#then need to do an action
df_orders.count()

26/03/06 13:11:25 WARN CacheManager: Asked to cache already cached data.
                                                                                

20000000

In [17]:
df_orders.rdd.getNumPartitions()

9

26/03/06 13:32:21 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_7 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_1 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_6 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_3 !
26/03/06 13:32:21 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_5 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_2 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_4 !
26/03/06 13:32:21 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_37_0 !
26/0

In [19]:
#disk_only

df_orders.unpersist()

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [20]:
df_orders.persist(StorageLevel.DISK_ONLY)

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [21]:
df_orders.count()

20000000

# MEMORY_ONLY_2 

In [22]:
#first clear cache
df_orders.unpersist()

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [23]:
#create cache
df_orders.persist(StorageLevel.MEMORY_ONLY_2)

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

In [25]:
df_orders.select(col('state')).show(5)

[Stage 12:>                                                         (0 + 1) / 1]

+----------------+
|           state|
+----------------+
|Himachal Pradesh|
|           Delhi|
|       Karnataka|
|           Delhi|
|         Haryana|
+----------------+
only showing top 5 rows



In [28]:
df_orders.unpersist()

DataFrame[order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int]

# Cache a table 

In [26]:
df_orders.createOrReplaceTempView('order_table')

In [29]:
spark.sql("CACHE TABLE order_table")


DataFrame[]

In [30]:
spark.sql("select * from order_table").show(5)

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [ ]:
#lazy
spark.sql("CACHE LAZY TABLE order_table")
spark.sql("select * from orders").write

# check table is cahe or not

In [31]:
spark.catalog.isCached('order_table')

True

#  uncache table

In [32]:
spark.sql("UNCACHE TABLE order_table")

DataFrame[]

In [33]:
spark.catalog.isCached('order_table')

False

# Best practices

# Catalyst Optimizer

In [34]:
#logical plan -- analysis exception
df_orders.select('status').show(5)

#status is not  a column ,so analysis exception---it comes under catalog whether column is there or not

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `status` cannot be resolved. Did you mean one of the following? [`state`, `price`, `order_status`, `quantity`, `order_date`].;
'Project ['status]
+- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv


In [36]:
df_orders.select('*').explain(True)

== Parsed Logical Plan ==
'Project [*]
+- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Analyzed Logical Plan ==
order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int
Project [order_id#137, customer_id#138, product_id#139, price#140, order_date#141, order_status#142, state#143, quantity#144]
+- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Optimized Logical Plan ==
Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Physical Plan ==
FileScan csv [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/orders_1gb.csv], Partition

In [37]:
df_orders.explain(True)

== Parsed Logical Plan ==
Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Analyzed Logical Plan ==
order_id: int, customer_id: int, product_id: int, price: float, order_date: date, order_status: string, state: string, quantity: int
Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Optimized Logical Plan ==
InMemoryRelation [order_id#137, customer_id#138, product_id#139, price#140, order_date#141, order_status#142, state#143, quantity#144], StorageLevel(memory, 1 replicas)
   +- FileScan csv [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/orders_1gb.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,price:float,order_da

# optimized logical plan

In [39]:
# it will optimize the query if it can be in optimised logical plan
df_orders.filter(col("order_status") =='PLACED')\
            .select("order_id","state")\
            .filter(col("state") == 'Goa')\
            .explain(True)

#Filter ((isnotnull(order_status#142) AND isnotnull(state#143)) AND ((order_status#142 = PLACED) AND (state#143 = Goa)))

== Parsed Logical Plan ==
'Filter ('state = Goa)
+- Project [order_id#137, state#143]
   +- Filter (order_status#142 = PLACED)
      +- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Analyzed Logical Plan ==
order_id: int, state: string
Filter (state#143 = Goa)
+- Project [order_id#137, state#143]
   +- Filter (order_status#142 = PLACED)
      +- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Optimized Logical Plan ==
Project [order_id#137, state#143]
+- Filter ((isnotnull(order_status#142) AND isnotnull(state#143)) AND ((order_status#142 = PLACED) AND (state#143 = Goa)))
   +- Relation [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] csv

== Physical Plan ==
*(1) Project [order_id#137, state#143]
+- *(1) Filter (((isnotnull(order_status#142) AND isnotnull(state#143

26/03/06 14:50:32 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 14:50:32 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 15:21:34 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 15:21:34 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 15:52:34 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 15:52:34 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 17:15:33 ERROR TaskSchedulerImpl: Lost executor 8 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/06 17:15:33 ERROR TaskSchedulerImpl: Lost executor 9 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 

# Join

In [41]:
df_orders.show(5) #1g file

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [42]:
#product table --12mb
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('product_name',StringType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products_12mb.csv',schema=product_schema_struct,header=True)

df_product.show(5)

+----------+------------+----------+
|product_id|product_name|cost_price|
+----------+------------+----------+
|         1|      168.37|      NULL|
|         2|      183.18|      NULL|
|         3|      155.47|      NULL|
|         4|      103.91|      NULL|
|         5|      103.64|      NULL|
+----------+------------+----------+
only showing top 5 rows



# inner join / shuffle sort merge join

In [44]:
df_orders.join(df_product,'product_id','inner')\
            .write.mode('overwrite')\
            .format('noop')\
            .save()

# broadcast hash join

In [46]:
df_orders.join(broadcast(df_product),'product_id','inner')\
            .write.mode('overwrite')\
            .format('noop')\
            .save()

In [49]:
df_bc_join = df_orders.join(broadcast(df_product),'product_id','inner')

df_bc_join.explain()

== Physical Plan ==
*(2) Project [product_id#139, order_id#137, customer_id#138, price#140, order_date#141, order_status#142, state#143, quantity#144, product_name#2310, cost_price#2311]
+- *(2) BroadcastHashJoin [product_id#139], [product_id#2309], Inner, BuildRight, false
   :- *(2) Filter isnotnull(product_id#139)
   :  +- FileScan csv [order_id#137,customer_id#138,product_id#139,price#140,order_date#141,order_status#142,state#143,quantity#144] Batched: false, DataFilters: [isnotnull(product_id#139)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/orders_1gb.csv], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,price:float,order_date:date,order_status:strin...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=621]
      +- *(1) Filter isnotnull(product_id#2309)
         +- FileScan csv [product_id#2309,product_name#2310,cost_price#2

 # shuffle hash join

In [51]:
# shuffle hash
df_orders.join(df_product.hint("shuffle_hash"), "product_id", "inner")\
        .write.mode("overwrite")\
        .format("noop")\
        .save()

# which one to use when

# Bucketing

In [54]:
spark.stop()

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_8_bucket")\
                    .config("spark.sql.warehouse.dir", "/data/spark-warehouse")\
                    .enableHiveSupport()\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 17:49:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [3]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_1gb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [4]:
#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('product_name',StringType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products.csv',schema=product_schema_struct,header=True)

df_product.show(5)

[Stage 1:>                                                          (0 + 1) / 1]

+----------+--------------------+----------+
|product_id|        product_name|cost_price|
+----------+--------------------+----------+
|         1|      Wireless Mouse|     120.0|
|         2| Mechanical Keyboard|     135.0|
|         3|       USB C Charger|     150.0|
|         4|        Laptop Stand|     165.0|
|         5|Noise Cancelling ...|     180.0|
+----------+--------------------+----------+
only showing top 5 rows



In [5]:
df_orders.write\
            .mode('overwrite')\
            .bucketBy(8,'product_id')\
            .sortBy('product_id')\
            .saveAsTable('orders_bucket')

#ideally 9parts should be there,but not in case of bucket

26/03/14 17:49:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/14 17:49:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/14 17:49:55 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/03/14 17:49:55 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.6
26/03/14 17:49:56 WARN HadoopFSUtils: The directory file:/data/spark-warehouse/orders_bucket was not found. Was it deleted very recently?
26/03/14 17:49:56 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/03/14 17:49:56 WARN FileUtils: File file:/data/spark-warehouse/orders_bucket does not exist; Force to delete it.
26/03/14 17:49:56 ERROR FileUtils: Failed to delete file:/data/spark-warehouse/orders_bucket
26/03/14 17:50:27 WARN SessionState:

In [6]:
df_product.write.mode('overwrite')\
            .bucketBy(8,'product_id')\
            .sortBy('product_id')\
            .saveAsTable('products_bucket')

26/03/14 17:54:59 WARN HadoopFSUtils: The directory file:/data/spark-warehouse/products_bucket was not found. Was it deleted very recently?
26/03/14 17:54:59 WARN FileUtils: File file:/data/spark-warehouse/products_bucket does not exist; Force to delete it.
26/03/14 17:54:59 ERROR FileUtils: Failed to delete file:/data/spark-warehouse/products_bucket


In [7]:
df_product.rdd.getNumPartitions()

1

In [18]:
df_orders.rdd.getNumPartitions()

9

In [ ]:
'''
spark.sql("
            select * from orders_bucket o inner join products_bucket p 
            on o.product_id=p.product_id
            ")\
            .write.format("noop")\
            .mode("overwrite")\
            .save()

'''




In [10]:
df_orders_bucket = spark.table("orders_bucket")
df_products_bucket = spark.table("products_bucket")


df_orders_bucket.join(df_products_bucket, "product_id", "inner")\
                .write.mode("overwrite")\
                .format("noop")\
                .save()

In [ ]:
#as we have 8 buckets-- 8 tasks will created if we are not going for broadcast join

In [15]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold","-1") #to disable broadcast join

In [16]:
#it will go for sort merge join
df_orders_bucket.join(df_products_bucket, "product_id", "inner")\
                .write.mode("overwrite")\
                .format("noop")\
                .save()

# Sort Aggregate vs Hash Aggregate

In [19]:
df_orders.groupBy(col('state')).agg(sum(col('quantity'))).write.mode('overwrite').format('noop').save()

#check sql/dataframe
#its going for hash agg in local and in the end